In [ ]:
pip install pandas numpy matplotlib seaborn scikit-learn

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

np.random.seed(42)
n_imoveis = 250

area = np.random.uniform(40, 220, n_imoveis)
quartos = np.random.choice([1, 2, 3, 4], size=n_imoveis, p=[0.25, 0.40, 0.25, 0.10])
distancia_metro = np.random.uniform(0.2, 5.0, n_imoveis) 

preco = (
    80000
    + (area * 5600) 
    + (quartos * 32000) 
    - (distancia_metro * 22000) 
    + np.random.normal(0, 35000, n_imoveis)
)

df_imoveis = pd.DataFrame({
    'Area_m2': np.round(area, 1),
    'Quartos': quartos,
    'Distancia_Metro_km': np.round(distancia_metro, 2),
    'Preco_Venda': np.round(preco, 2)
})

print("Base de dados criada com sucesso! Primeiras linhas:")
display(df_imoveis.head())

Base de dados criada com sucesso! Primeiras linhas:


,Area_m2,Quartos,Distancia_Metro_km,Preco_Venda
0,107.4,2,3.55,615660.04
1,211.1,2,2.77,1305339.64
2,171.8,3,1.69,1112611.15
3,147.8,2,4.11,866575.92
4,68.1,1,3.49,438706.54


In [ ]:
#Parte 1
#1
df_imoveis[['Preco_Venda', 'Area_m2']].describe().loc[['mean', 'min', 'max']]

In [ ]:
#2
matriz_corr = df_imoveis.corr()
plt.figure(figsize=(8, 6))
sns.heatmap(matriz_corr, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Mapa de Calor da Correlação')
plt.show()

#PERGUNTA
#A variável que possui correlação mais forte como Preco_Venda é Area_m2. Possui relação positiva. 

In [ ]:
#3 
sns.scatterplot(data=df_imoveis, x="Area_m2", y="Preco_Venda")
plt.title("Dispersão: Área vs Preço de Venda")
plt.xlabel("Área (m²)")
plt.ylabel("Preço de Venda")
plt.show()

In [ ]:
#Parte 2 
X = df_imoveis[['Area_m2']]
y = df_imoveis['Preco_Venda']


X_treino, X_teste, y_treino, y_teste = train_test_split(
    X, y, test_size=0.2, random_state=42
)

modelo = LinearRegression()
modelo.fit(X_treino, y_treino)

coef_angular = modelo.coef_[0]
intercepto = modelo.intercept_

print(f'Coeficiente Angular (Inclinação): {coef_angular:.2f}')
print(f'Intercepto Linear: {intercepto:.2f}\n')

y_pred = modelo.predict(X_teste)

r2 = r2_score(y_teste, y_pred)
mae = mean_absolute_error(y_teste, y_pred)
rmse = np.sqrt(mean_squared_error(y_teste, y_pred))

print('Métricas de Avaliação:')
print(f'R²: {r2:.4f}')
print(f'MAE: R$ {mae:.2f}')
print(f'RMSE: R$ {rmse:.2f}\n')

plt.figure(figsize=(10, 6))
plt.scatter(
    X_teste, y_teste, color='purple', alpha=0.6, label='Dados Reais (Teste)'
)
plt.plot(
    X_teste,
    y_pred, 
    color='blue',
    linewidth=2,
    label='Reta de Regressão (Previsto)',
)
plt.title('Regressão Linear Baseline: Área vs Preço de Venda')
plt.xlabel('Área (m²)')
plt.ylabel('Preço de Venda (R$)')
plt.legend()
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

In [27]:
#Parte 3

X_multi = df_imoveis[['Area_m2', 'Quartos', 'Distancia_Metro_km']]
y = df_imoveis['Preco_Venda']

X_train, X_test, y_train, y_test = train_test_split(X_multi, y, test_size=0.2, random_state=42)

modelo_multi = LinearRegression()
modelo_multi.fit(X_train, y_train)

df_coeficientes = pd.DataFrame({
    'Variavel': X_multi.columns,
    'Impacto_no_Preco_R$': modelo_multi.coef_
})
print('Coeficientes do Modelo')
print(df_coeficientes)

Coeficientes do Modelo
             Variavel  Impacto_no_Preco_R$
0             Area_m2          5599.151708
1             Quartos         31655.267604
2  Distancia_Metro_km        -21563.587421


In [ ]:
#Parte 4

novos_imoveis = pd.DataFrame({
    'Area_m2': [65.0, 110.0, 180.0],
    'Quartos': [2, 3, 4],
    'Distancia_Metro_km': [0.5, 1.8, 4.2]
})

precos_sugeridos = modelo_multi.predict(novos_imoveis)

novos_imoveis['Preco_Sugerido_R$'] = precos_sugeridos

resultado = novos_imoveis.copy()

resultado['Preco_Sugerido_R$'] = resultado['Preco_Sugerido_R$'].apply(
    lambda x: f"R$ {x:,.2f}".replace(',', 'X').replace('.', ',').replace('X', '.')
)

display(resultado)



PARTE 5

Questão 5
O corretor não deve confiar 100% no valor previsto. O MAE indica que o erro médio é de aproximadamente R$ 28.000. Portanto, o modelo deve ser usado como referência ou faixa de negociação, junto com a análise do corretor e outras características do imóvel.

Questão 2
Uma cobertura de 800 m² está muito fora da faixa dos dados usados no treinamento que é de aproximadamente 40 a 220 m². A previsão seria um desvio e poderia ser pouco confiável. Portanto, não é seguro utilizar o modelo para dados muito distantes dos observados no treinamento.

Questão 3
Como Bairro é uma variável categórica, pode-se utilizar One-Hot Encoding, criando uma coluna para cada bairro.

df = pd.get_dummies(df, columns=['Bairro'], dtype=int)

Assim, alguns bairros seriam transformados em colunas com valores 0 e 1, podendo ser utilizados na Regressão Linear.
